In [4]:
import yfinance as yf
import baostock as bs
import pandas as pd
from tqdm import tqdm
import time
import os
import random
import akshare as ak

In [5]:
import os
proxy = 'http://127.0.0.1:7890' # 代理设置，此处修改
os.environ['HTTP_PROXY'] = proxy
os.environ['HTTPS_PROXY'] = proxy

In [3]:
# 彻底禁掉代理（见上一节）
for v in ("HTTP_PROXY","HTTPS_PROXY","http_proxy","https_proxy"):
    os.environ.pop(v, None)

# 获取港股

In [3]:
# 1. 拉取港股实时列表
df_hk = ak.stock_hk_spot_em()
# akshare 返回的代码列叫 '代码'
# 格式化成 yfinance 识别的形式："0700" → "0700.HK"
tickers = df_hk['代码'].apply(lambda x: f"{x}.HK").tolist()

# （可选）过滤掉少量格式不标准的，确保都是 4 位数字开头
tickers = [t for t in tickers if t.split('.')[0].isdigit()]

print(f"共拿到 {len(tickers)} 只港股代码示例：", tickers[:5])
codes = df_hk['代码'].tolist()

  0%|          | 0/45 [00:00<?, ?it/s]

共拿到 4583 只港股代码示例： ['01152.HK', '01667.HK', '02442.HK', '08537.HK', '00202.HK']


In [4]:
tickers

['01152.HK',
 '01667.HK',
 '02442.HK',
 '08537.HK',
 '00202.HK',
 '08249.HK',
 '00602.HK',
 '01080.HK',
 '02400.HK',
 '01967.HK',
 '00767.HK',
 '00265.HK',
 '01228.HK',
 '01938.HK',
 '06113.HK',
 '08340.HK',
 '02415.HK',
 '00508.HK',
 '01237.HK',
 '02170.HK',
 '08223.HK',
 '01797.HK',
 '08100.HK',
 '02182.HK',
 '02427.HK',
 '02350.HK',
 '01812.HK',
 '00348.HK',
 '06998.HK',
 '08277.HK',
 '00323.HK',
 '01968.HK',
 '01672.HK',
 '06622.HK',
 '00093.HK',
 '02558.HK',
 '01428.HK',
 '02500.HK',
 '01432.HK',
 '01286.HK',
 '08366.HK',
 '08153.HK',
 '00568.HK',
 '06918.HK',
 '00865.HK',
 '00899.HK',
 '01631.HK',
 '09606.HK',
 '08176.HK',
 '01741.HK',
 '01947.HK',
 '02522.HK',
 '06855.HK',
 '08456.HK',
 '00075.HK',
 '02696.HK',
 '02168.HK',
 '00915.HK',
 '02340.HK',
 '08232.HK',
 '00983.HK',
 '01473.HK',
 '03728.HK',
 '00619.HK',
 '09968.HK',
 '01150.HK',
 '00799.HK',
 '08480.HK',
 '00326.HK',
 '02157.HK',
 '01020.HK',
 '09930.HK',
 '01877.HK',
 '08106.HK',
 '00320.HK',
 '08040.HK',
 '01022.HK',

In [8]:
# —— 2. 定义时间区间和频率 —— 
start_date = "2025-07-01"
end_date   = "2025-08-01"
interval   = "1d"    # 日线

In [9]:
# 添加重试机制
# 安全下载收盘价函数
def safe_download_close(ticker, max_retries=1, wait_time=1):
    for attempt in range(max_retries):
        try:
            data = yf.download(
                tickers=ticker,
                start = start_date,
                end = end_date,      # 最近一年
                interval="1d",      # 日线
                progress=False,
                auto_adjust=True    # 自动复权
            )
            if not data.empty:
                # 只保留收盘价
                return data[['Close']]
        except Exception as e:
            print(f"[错误] {ticker} 第 {attempt+1}/{max_retries} 次尝试失败: {str(e)}")
        
        time.sleep(wait_time)
    
    print(f"[失败] 无法获取 {ticker} 的数据")
    return pd.DataFrame()


# 使用安全下载函数
ticker_symbol = "08025.HK"
stock_data = safe_download_close(ticker_symbol)


1 Failed download:
['08025.HK']: YFPricesMissingError('possibly delisted; no price data found  (1d 2025-07-01 -> 2025-08-01)')


[失败] 无法获取 08025.HK 的数据


In [10]:
stock_data

""


In [14]:
# 存储所有收盘价数据
all_close = pd.DataFrame()

In [12]:
tickers[1][1:]

'1667.HK'

In [ ]:
# 遍历下载
for ticker in tqdm(tickers, desc="下载港股收盘价"):
    if ticker.startswith("0"):
        ticker = ticker[1:]
    close_df = safe_download_close(ticker)
    if not close_df.empty:
        # 重命名列为股票代码
        close_df.rename(columns={"Close": ticker}, inplace=True)
        
        # 合并数据（按日期对齐）
        if all_close.empty:
            all_close = close_df
        else:
            all_close = all_close.join(close_df, how='outer')

In [15]:
tickers

['01152.HK',
 '01667.HK',
 '02442.HK',
 '08537.HK',
 '00202.HK',
 '08249.HK',
 '00602.HK',
 '01080.HK',
 '02400.HK',
 '01967.HK',
 '00767.HK',
 '00265.HK',
 '01228.HK',
 '01938.HK',
 '06113.HK',
 '08340.HK',
 '02415.HK',
 '00508.HK',
 '01237.HK',
 '02170.HK',
 '08223.HK',
 '01797.HK',
 '08100.HK',
 '02182.HK',
 '02427.HK',
 '02350.HK',
 '01812.HK',
 '00348.HK',
 '06998.HK',
 '08277.HK',
 '00323.HK',
 '01968.HK',
 '01672.HK',
 '06622.HK',
 '00093.HK',
 '02558.HK',
 '01428.HK',
 '02500.HK',
 '01432.HK',
 '01286.HK',
 '08366.HK',
 '08153.HK',
 '00568.HK',
 '06918.HK',
 '00865.HK',
 '00899.HK',
 '01631.HK',
 '09606.HK',
 '08176.HK',
 '01741.HK',
 '01947.HK',
 '02522.HK',
 '06855.HK',
 '08456.HK',
 '00075.HK',
 '02696.HK',
 '02168.HK',
 '00915.HK',
 '02340.HK',
 '08232.HK',
 '00983.HK',
 '01473.HK',
 '03728.HK',
 '00619.HK',
 '09968.HK',
 '01150.HK',
 '00799.HK',
 '08480.HK',
 '00326.HK',
 '02157.HK',
 '01020.HK',
 '09930.HK',
 '01877.HK',
 '08106.HK',
 '00320.HK',
 '08040.HK',
 '01022.HK',

In [17]:
all_close

Price,1152.HK,1667.HK,2442.HK,8537.HK,0202.HK,8249.HK,0602.HK,1080.HK,2400.HK,1967.HK,...,0286.HK,0279.HK,0274.HK,0223.HK,0195.HK,0176.HK,0102.HK,0059.HK,0009.HK,0007.HK
Ticker,1152.HK,1667.HK,2442.HK,8537.HK,0202.HK,8249.HK,0602.HK,1080.HK,2400.HK,1967.HK,...,0286.HK,0279.HK,0274.HK,0223.HK,0195.HK,0176.HK,0102.HK,0059.HK,0009.HK,0007.HK
Date,,,,,,,,,,,,,,,,,,,,,
2025-07-02,0.073,0.13,1.30,0.167,0.055,0.139,0.040,0.038,49.000000,0.405,...,1.31,0.92,0.49,0.109,0.285,0.013,0.047,0.01,0.013,0.032
2025-07-03,0.073,0.13,1.35,0.180,0.055,0.138,0.043,0.037,48.400002,0.400,...,1.31,0.82,0.49,0.109,0.285,0.013,0.047,0.01,0.013,0.032
2025-07-04,0.073,0.13,1.37,0.187,0.058,0.138,0.044,0.039,49.849998,0.385,...,1.31,0.78,0.49,0.109,0.285,0.013,0.047,0.01,0.013,0.032
2025-07-07,0.073,0.13,1.38,0.187,0.058,0.098,0.044,0.037,51.000000,0.340,...,1.31,0.80,0.49,0.109,0.285,0.013,0.047,0.01,0.013,0.032
2025-07-08,0.073,0.13,1.38,0.170,0.058,0.100,0.044,0.040,51.299999,0.355,...,1.31,0.80,0.49,0.109,0.285,0.013,0.047,0.01,0.013,0.032
2025-07-09,0.073,0.13,1.42,0.170,0.057,0.100,0.044,0.042,50.900002,0.395,...,1.31,0.84,0.49,0.109,0.285,0.013,0.047,0.01,0.013,0.032
2025-07-10,0.073,0.13,1.42,0.169,0.060,0.100,0.044,0.060,50.049999,0.465,...,1.31,0.82,0.49,0.109,0.285,0.013,0.047,0.01,0.013,0.032
2025-07-11,0.073,0.13,1.42,0.171,0.059,0.102,0.044,0.054,48.849998,0.485,...,1.31,0.86,0.49,0.109,0.285,0.013,0.047,0.01,0.013,0.032


In [19]:
all_close = all_close.T

In [21]:
all_close.to_csv('../data/hongkong.csv')

# 获取A股

In [3]:
lg = bs.login()
if lg.error_code != "0":
    raise RuntimeError("登录失败：" + lg.error_msg)

# 2. 拉取证券基本资料（只需要 code 和 name，也可以带其他列）
rs = bs.query_stock_basic()
# 把结果逐条读到列表里
data_list = []
while rs.error_code == '0' and rs.next():
    data_list.append(rs.get_row_data())

# 3. 转成 DataFrame，并清理列名空格
df = pd.DataFrame(data_list, columns=rs.fields)
df.columns = [c.strip() for c in df.columns]

print("所有证券样例：")
print(df.head(), "\n", df.columns.tolist())

# 4. 只保留 A 股股票
#    type == '1' 表示“股票”；status == '1' 表示“在市”
df_a = df[(df['type']   == '1') &
          (df['status'] == '1')].reset_index(drop=True)

print(f"共计 {len(df_a)} 只在市的 A 股")
print(df_a.head())

bs.logout()

login success!
所有证券样例：
        code code_name     ipoDate outDate type status
0  sh.000001    上证综合指数  1991-07-15            2      1
1  sh.000002    上证A股指数  1992-02-21            2      1
2  sh.000003    上证B股指数  1992-08-17            2      1
3  sh.000004   上证工业类指数  1993-05-03            2      1
4  sh.000005   上证商业类指数  1993-05-03            2      1 
 ['code', 'code_name', 'ipoDate', 'outDate', 'type', 'status']
共计 5183 只在市的 A 股
        code code_name     ipoDate outDate type status
0  sh.600000      浦发银行  1999-11-10            1      1
1  sh.600004      白云机场  2003-04-28            1      1
2  sh.600006      东风股份  1999-07-27            1      1
3  sh.600007      中国国贸  1999-03-12            1      1
4  sh.600008      首创环保  2000-04-27            1      1
logout success!


In [ ]:
df_a = df_a['code'].tolist()

In [ ]:
df_a

In [5]:
# 3. 定义拉取区间
start_date = "2023-07-01"
end_date   = "2023-08-31"

records = []
batch_size = 1   # 每1只重连一次

for i, code in enumerate(tqdm(df_a, desc="拉取收盘价")):
    # 每 batch_size 只断开并重连一次
    if i and i % batch_size == 0:
        bs.logout()
        time.sleep(0.1)      # 等待底层完全断开
        bs.login()         # 重新登录

    # 拉取并重试
    for attempt in range(3):
        try:
            rs = bs.query_history_k_data_plus(
                code,
                "date,close",
                start_date=start_date,
                end_date=end_date,
                frequency="d",
                adjustflag="3"
            )
            df = rs.get_data()
            if not df.empty:
                df["code"] = code
                records.append(df)
            break
        except Exception as e:
            print(f"[{code}] 第 {attempt+1} 次失败：{e}，等待后重试")
            time.sleep(0.1)
    time.sleep(0.1)  # **1 秒/只**，极限限速

# 4. 合并 & pivot
df_result = pd.concat(records, ignore_index=True)
df_wide = df_result.pivot(index="code", columns="date", values="close")
df_wide.columns = [pd.to_datetime(d).strftime("%m-%d")
                   for d in df_wide.columns]


拉取收盘价:   0%|          | 1/5183 [00:00<08:43,  9.89it/s]

股票代码应为9位，请检查。格式示例：sh.600000。
logout success!


拉取收盘价:   0%|          | 2/5183 [00:01<58:07,  1.49it/s]

login success!
logout success!


拉取收盘价:   0%|          | 3/5183 [00:01<58:23,  1.48it/s]

...

logout success!
login success!
股票代码应为9位，请检查。格式示例：sh.600000。


拉取收盘价:   0%|          | 5/5183 [00:02<48:08,  1.79it/s]

logout success!
login success!
股票代码应为9位，请检查。格式示例：sh.600000。


拉取收盘价:   0%|          | 6/5183 [00:03<48:43,  1.77it/s]


ValueError: No objects to concatenate

In [ ]:
df_wide.to_csv('../data/data.csv')

In [ ]:
# 5. 登出
bs.logout()

# 获取A股单只股票价格

In [20]:
# 光线传媒：300251.SZ
df = yf.download("300251.SZ",
                 start="2025-01-14",
                 end="2025-02-18",   # end为开区间，写到次日以包含 02-17
                 interval="1d",
                 progress=False)

print(df["Close"])  # 每天的收盘价

C:\Users\Administrator\AppData\Local\Temp\ipykernel_4064\1028534116.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download("300251.SZ",


Ticker      300251.SZ
Date                 
2025-01-14   8.890718
2025-01-15   8.801711
2025-01-16   8.841269
2025-01-17   9.197294
2025-01-20   9.276411
2025-01-21   9.256632
2025-01-22   9.335748
2025-01-23   9.414865
2025-01-24   9.395085
2025-01-27   9.424754
2025-02-05  11.313661
2025-02-06  13.350911
2025-02-07  13.805831
2025-02-10  16.565020
2025-02-11  19.878023
2025-02-12  23.853628
2025-02-13  28.620398
2025-02-14  34.346455
2025-02-17  29.332447


In [21]:
df = df["Close"]
df.to_csv('../data/nezha.csv',index=False)